# Titanic 数据处理实战

流程：**读取 → 预览 → 筛选 → 分组分析 → 保存结果**

---
## Step 1：读取数据

In [ ]:
import pandas as pd

url = "practice.csv"
df = pd.read_csv(url)
df.head()

---
## Step 2：检查数据 — 多少行多少列，有没有空值

In [ ]:
# 行列数
print(f"行数: {df.shape[0]}, 列数: {df.shape[1]}")
df.shape

In [ ]:
# 每列的数据类型和非空数量
df.info()

In [ ]:
# 每列缺失值统计
df.isnull().sum()

> **发现**：Age 缺 177 个，Cabin 缺 687 个，Embarked 缺 2 个。Cabin 缺太多，后面分析可以忽略这列。

In [ ]:
# 数值列的统计摘要
df.describe()

---
## Step 3：数据清洗 — 处理缺失值

In [ ]:
# 用均值填充年龄
df["Age"] = df["Age"].fillna(df["Age"].mean())

# Embarked 只缺 2 个，用众数填
df["Embarked"] = df["Embarked"].fillna(df["Embarked"].mode()[0])

# Cabin 缺太多，直接删掉这列
df = df.drop("Cabin", axis=1)

# 验证：确认没有缺失值了
df.isnull().sum()

---
## Step 4：筛选 — 挑出满足条件的数据

In [ ]:
# 筛选 1：未成年乘客（< 18 岁）
children = df[df["Age"] < 18]
children.head()

In [ ]:
# 筛选 2：一等舱的女性乘客
first_class_female = df[(df["Pclass"] == 1) & (df["Sex"] == "female")]
first_class_female

In [ ]:
# 筛选 3：幸存且年龄 > 60 的乘客
old_survivors = df[(df["Survived"] == 1) & (df["Age"] > 60)]
old_survivors

In [ ]:
# 用 query 写法（更直观）
df.query("Pclass == 1 and Sex == 'female'")

---
## Step 5：分组分析 — 按类别算平均值

In [ ]:
# 按性别分组，看存活率
df.groupby("Sex")["Survived"].mean()

In [ ]:
# 按舱位等级分组，看平均年龄和存活率
df.groupby("Pclass").agg({
    "Age": "mean",
    "Survived": "mean",
    "PassengerId": "count"  # 数人头
}).rename(columns={"Pas"
"sengerId": "count"})

In [ ]:
# 按登船港口 + 性别分组，看存活率
df.groupby(["Embarked", "Sex"])["Survived"].mean().round(3)

In [ ]:
# 带家人的乘客 vs 独自一人的乘客，存活率对比
df["FamilySize"] = df["SibSp"] + df["Parch"]
df["IsAlone"] = (df["FamilySize"] == 0).astype(int)
df.groupby("IsAlone")["Survived"].mean()

---
## Step 6：保存处理后的数据

In [ ]:
# 保存清洗后的数据
df.to_csv("new_pandas_practice.csv", index=False)
print("已保存 titanic_cleaned.csv")
print(f"最终数据: {df.shape[0]} 行, {df.shape[1]} 列")

---
## Step 6（附加）：可视化 — 把关键结论画出来

In [ ]:
import matplotlib.pyplot as plt

# 设置中文字体（否则中文会变成方块）
plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# 图 1：不同舱位的存活率（柱状图）
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

survival_by_class = df.groupby("Pclass")["Survived"].mean()
axes[0].bar(survival_by_class.index, survival_by_class.values, color=["gold", "silver", "sandybrown"])
axes[0].set_title("Survival Rate by Class")
axes[0].set_xlabel("Pclass")
axes[0].set_ylabel("Survival Rate")
for i, v in enumerate(survival_by_class.values):
    axes[0].text(i + 1, v + 0.02, f"{v:.2f}", ha="center")

# 图 2：性别存活率（饼图）
survival_by_sex = df.groupby("Sex")["Survived"].mean()
axes[1].pie(survival_by_sex.values, labels=survival_by_sex.index,
            autopct="%1.1f%%", colors=["#ff9999", "#66b3ff"])
axes[1].set_title("Survival Rate by Sex")

plt.tight_layout()
plt.savefig("titanic_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 小结

这个流程你后面做任何数据分析项目都会重复：

```
pd.read_csv()         → 读进来
.shape / .info()      → 看结构
.isnull().sum()       → 查缺失
.fillna() / dropna()  → 清洗
df[条件]               → 筛选
.groupby().mean()     → 分组分析
.to_csv()             → 存结果
plt.bar() / plt.pie()  → 画出来
```

**Titanic 数据的核心发现：**
- 女性存活率 74%，男性仅 19%
- 一等舱存活率 63%，三等舱 24%
- 有家人同行 vs 独自一人，存活率有差异